# 11 · 长期记忆：可追溯不等于可信

记忆是下一任务的输入，因此比“把上一轮文字存下来”多一个准入问题：来源可核对吗，范围仍适用吗，用户更正后哪一条生效？
本章把一条 task-list 字段经验走完提取、合并、投影、更正和遗忘。SQLite 的事务负责让状态与操作回执一起提交；
来源校验只证明引文存在，不能证明候选结论被引文支持。后者必须用反例直接暴露。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [M01 · start_memories_startup_task](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/memories/write/src/start.rs#L20-L67)：后台流水线由会话启动触发；ephemeral、MemoryTool关闭、非root排除。V1+dual_write调度V1/V2，V2只调度V2。
- [M21 · read-path developer prompt](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/memories/templates/memories/read_path_v2.md#L3-L20)：V2读prompt明确summary是历史上下文，重要/易变结论需核对当前source；显式remember/forget/correction通过ad-hoc note交给合并。
- [M22 · add_ad_hoc_note](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/memories/src/local/ad_hoc_note.rs#L12-L39)：ad-hoc note在extensions/ad_hoc/notes中create_new写入，不覆盖已有note，不直接修改生成summary。


## 实验 1 · 合并证据，而非重复注入


同一事实来自两个会话时，应合并支持证据，保持一个活跃记忆。`source_version` 是实际文本摘要，
`applies_at` 是教学版显式版本边界。先真实写 SQLite，再关闭重开，确认不是内存字典造成的假持久化。


In [2]:
memory = load_example('11', 'ledger')
from dataclasses import replace
import json
run_dir = fresh_run_dir('ch11')
path = run_dir/'memory.sqlite'
ledger = memory.Ledger(path)
quote = 'Task records use status; filter_tasks reads status.'
body = 'Task: repair task-list\n' + quote + '\nStatus case passed.'
version = ledger.put_source('session-A', body)
candidate = memory.Candidate('task-list', 'status-field', quote,
    'session-A', version, 2, 2, quote, 'repo-v1')
a = ledger.merge('extract-A', candidate)
v2 = ledger.put_source('session-B', body + '\nRelated task completed.')
b = ledger.merge('extract-B', replace(candidate, source='session-B', source_version=v2))
assert a['id'] == b['id'] and b['status'] == 'merged_support'
ledger.close()
ledger = memory.Ledger(path)
projection = ledger.project('task-list', 'repo-v1', 'filter', 10000)
assert len(projection['ids']) == 1
assert len(json.loads(projection['context'])['evidence']) == 2
print({'committed_id': a['id'], 'evidence_count': 2, 'database': path.name})


{'committed_id': 1, 'evidence_count': 2, 'database': 'memory.sqlite'}


## 实验 2 · 改版本与预算，观察投影边界


记忆存在，不代表每个请求都应携带它。版本变化返回 `needs_revalidation`，预算不足返回 `budget`。
这里按整条 UTF-8 序列化字节计费，不声称等价于 tokenizer。把 `BUDGET` 改成 0、300、10000，观察整条选入或排除。


In [3]:
BUDGET = 10000
current = ledger.project('task-list', 'repo-v1', '', BUDGET)
changed = ledger.project('task-list', 'repo-v2', '', 10000)
other = ledger.project('other-project', 'repo-v1', '', 10000)
small = ledger.project('task-list', 'repo-v1', '', 0)
assert current['bytes'] <= BUDGET
assert changed['ids'] == [] and changed['excluded'][0]['reason'] == 'needs_revalidation'
assert other['ids'] == []
assert small['excluded'][0]['reason'] == 'budget'
print({'budget_projection': current['ids'], 'version_exclusion': changed['excluded']})


{'budget_projection': [1], 'version_exclusion': [{'id': 1, 'reason': 'needs_revalidation'}]}


## 实验 3 · 故意让“有引用的假结论”入库


伪造引用应被程序拒绝；把真实引用配给不受支持的结论却能通过本内核。这是刻意保留的失败反例，
说明来源验证与语义验证是两个责任。该反例单独使用错误 key，避免污染后面的字段更正实验。


In [4]:
with expect.assertRaisesRegex(memory.Rejected, 'source_quote_mismatch'):
    ledger.merge('bad-quote', replace(candidate, quote='never present in source'))
unsupported = replace(candidate, key='unsupported-example',
    text='All task-list operations always use Redis.')
accepted = ledger.merge('cited-but-untrue', unsupported)
assert accepted['status'] == 'active'
assert 'Redis' in ledger.project('task-list', 'repo-v1', 'Redis', 10000)['context']
ledger.forget('remove-counterexample', 'task-list', 'unsupported-example',
              'Teaching fixture: withdraw unsupported claim.')
print('反例已观测：来源存在并不能推出候选语义正确。')


反例已观测：来源存在并不能推出候选语义正确。


## 实验 4 · 冲突暂存，更正显式替换


新提取与旧条目冲突时，默认保留旧活跃值，不能用“较新”直接覆盖。更正接口使用旧记录 ID 做比较，
防止另一更正已提交后继续写旧前提。实验中的授权理由是合成用户指令；本内核不验证真实用户身份。


In [5]:
new_quote = 'Correction: schema v2 uses state; filter_tasks reads state.'
v3 = ledger.put_source('session-C', new_quote)
new_candidate = replace(candidate, text=new_quote, source='session-C',
    source_version=v3, start=1, end=1, quote=new_quote, applies_at='repo-v2')
conflict = ledger.merge('extract-conflict', new_candidate)
assert conflict['status'] == 'conflict'
assert ledger.project('task-list', 'repo-v1', '', 10000)['ids'] == [a['id']]
corrected = ledger.correct('explicit-correction', a['id'], new_candidate,
                           'Synthetic user: use schema v2 from now on.')
assert corrected['supersedes'] == a['id']
with expect.assertRaisesRegex(memory.Rejected, 'active_version_conflict'):
    ledger.correct('stale-correction', a['id'], new_candidate, 'Repeat with stale version')
assert 'schema v2' in ledger.project('task-list', 'repo-v2', '', 10000)['context']
print(corrected)


{'status': 'corrected', 'id': 4, 'supersedes': 1}


## 实验 5 · 取消回滚、丢回执重放与遗忘屏障


事务提交前抛取消，应连回执一起回滚；提交后客户端丢回执，再用相同操作 ID 读取同一结果。
遗忘阻止未来投影与新提取复活，但不抹去已经发送的上下文，也不等同于删除来源档案。


In [6]:
class TeachingCancel(BaseException):
    pass

def cancel_before_commit():
    raise TeachingCancel()

pending = replace(candidate, key='cancel-case')
with expect.assertRaises(TeachingCancel):
    ledger.merge('retry-op', pending, barrier=cancel_before_commit)
assert ledger.db.execute("SELECT count(*) FROM operations WHERE op='retry-op'").fetchone()[0] == 0
first = ledger.merge('retry-op', pending)
ledger.close()
ledger = memory.Ledger(path)
assert ledger.merge('retry-op', pending) == first
assert ledger.db.execute("SELECT count(*) FROM memories WHERE key='cancel-case'").fetchone()[0] == 1
old_context = ledger.project('task-list', 'repo-v2', '', 10000)['context']
ledger.forget('forget-field', 'task-list', 'status-field', 'Synthetic user requested forgetting.')
ledger.close()
ledger = memory.Ledger(path)
assert ledger.merge('new-extraction-after-forget', new_candidate)['status'] == 'blocked_by_forgetting'
assert ledger.project('task-list', 'repo-v2', 'schema v2', 10000)['context'] == ''
assert 'schema v2' in old_context
ledger.close()
print('提交前取消已回滚；提交后重放未重复；遗忘屏障跨重启保留。')


提交前取消已回滚；提交后重放未重复；遗忘屏障跨重启保留。


## 你现在可以解释什么

能分别解释来源真实性、结论正确性、当前适用性；能指出提交点与遗忘的真实作用域。这个实验没有实现模型提取器、语义检索或源码的后台合并流水线。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
